# Hydrogen supply under a carbon cap and a shortage penalty

Explore a cost–service trade-off from the legacy Houston flexibility study. Make a capacity and dispatch decision under a fixed carbon budget. A penalty per unit of unmet hydrogen demand lets the model price the value of serving more of the requested load.

This compact example ports the penalty-sweep workflow as a linear planning problem. It uses illustrative cost and availability inputs and compares operating outcomes after solving each penalty case.

## Set up the model

The system has two routes. Reformation is cheaper but emits CO₂, while renewable electrolysis has higher production cost. Add an explicit shortage variable to close the delivery target in every period. The carbon cap limits total emissions over the four periods.

In [1]:
from math import isclose

import pandas as pd
from energia import Model, Periods, Process, Resource
from gana import V, inf

demand = [5, 8, 8, 5]
wind_limit = [4, 3, 2, 4]
emissions_cap = 100
emissions_per_unit = 9.5


def plan_supply(penalty):
    m = Model(f"flexibility_{penalty}")
    m.block = Periods()
    m.day = len(demand) * m.block
    m.declare(Resource, ["gas", "wind_power", "hydrogen"])
    _ = m.gas.consume(m.block) >= 0
    _ = m.wind_power.consume(m.block) <= wind_limit
    delivered = m.hydrogen.release(m.block)
    _ = delivered >= 0

    m.reformer = Process(m.hydrogen == -3.76 * m.gas)
    m.electrolyzer = Process(m.hydrogen == -m.wind_power)
    for route in (m.reformer, m.electrolyzer):
        _ = route.capacity(m.day) <= 8
        _ = route.operate(m.block) <= 1
        route.locate()

    reforming = m.reformer.operate(m.block).V()
    renewables = m.electrolyzer.operate(m.block).V()
    shipments = list(delivered.V())
    m.program.shortage = V(*delivered.I)
    for number, (served, gap, requested) in enumerate(zip(shipments, m.program.shortage, demand, strict=True)):
        setattr(m.program, f"delivery_{number}", served + gap == requested)
    emissions = emissions_per_unit * sum(reforming)
    m.program.emissions_limit = emissions <= emissions_cap

    shortage = sum(m.program.shortage)
    supply_cost = (
        60 * m.reformer.capacity(m.day).V()[0] + 75 * m.electrolyzer.capacity(m.day).V()[0]
        + 40 * sum(reforming) + 120 * sum(renewables)
    )
    m.program.total_cost = inf(supply_cost + penalty * shortage)
    m.opt()

    served = sum(delivered.output(aslist=True))
    unmet = sum(m.program.shortage.output(aslist=True))
    assert isclose(served + unmet, sum(demand), abs_tol=1e-6)
    return {
        "Supply cost": m.program.active_objective.X - penalty * unmet,
        "Unmet hydrogen": unmet,
        "CO₂": emissions_per_unit * sum(reforming.output(aslist=True)),
    }

## Sweep the shortage penalty

In [2]:
results = pd.DataFrame({penalty: plan_supply(penalty) for penalty in [20, 100, 200, 400]}).T
results.index.name = "Shortage penalty"
assert results["Unmet hydrogen"].is_monotonic_decreasing
assert results["Unmet hydrogen"].iloc[-1] < results["Unmet hydrogen"].iloc[0]
assert results["CO₂"].max() <= emissions_cap + 1e-6
results.round(2)

,Supply cost,Unmet hydrogen,CO₂
Shortage penalty,,,
20,0.00,26.00,0.0
100,578.95,15.47,100.0
200,2536.84,2.47,100.0
400,2536.84,2.47,100.0
